In [29]:
import os
os.chdir("/users/ffbence/dose")

In [30]:
from scripts.models import DoTA_based
from omegaconf import OmegaConf
import torch

In [31]:
cfg = OmegaConf.load('configs/default_config.yaml')
cfg = cfg['atlasz']
cfg = OmegaConf.merge(cfg, OmegaConf.load('configs/db_config.yaml'))

In [32]:
OmegaConf.save(cfg, 'checkpoints/atlasz_Stronger_idd_DoTA/cfg.yaml')

## Test model loading

In [10]:
checkpointsRoot ="checkpoints"
experiment ="atlasz_Stronger_idd_DoTA"
version = "last"
#cfg = OmegaConf.load(os.path.join(checkpointsRoot, experiment,"cfg.yaml"))
model = DoTA_based(**cfg['dotakwgs'])
model_state_dict = torch.load(os.path.join(checkpointsRoot, experiment, version)+".ckpt")["state_dict"]
model_state_dict = {k.replace("model.", ""): v for k, v in model_state_dict.items()}
model.load_state_dict(model_state_dict)

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True


<All keys matched successfully>

In [11]:
from scripts.utilities import DoseCallWrapper
from monai.transforms import Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, ScaleIntensityRanged, Spacingd, ResizeWithPadOrCropd, EnsureTyped
from scripts.metaembedder import InjectGaussianBeamPriord, InjectEnergyDepositionFieldd
from scripts.data_loader import ExtractSlabsAroundZ

In [12]:
infer_transforms = Compose([
    LoadImaged(keys=["ct", "gt_dose"]),
    EnsureChannelFirstd(keys=["ct"]),
    ScaleIntensityRanged(keys=['ct'], a_min=cfg['ct_min'], a_max=cfg['ct_max'], b_min=0.0, b_max=1.0, clip=True),
    ExtractSlabsAroundZ(keys=["ct",], source_key="ray_source", slice_radius=15),
    Spacingd(keys=["ct",], pixdim=cfg['pixdim'], mode='trilinear'),
    ResizeWithPadOrCropd(keys=["ct"], spatial_size=cfg['roi_size']),
    InjectGaussianBeamPriord(keys=['geometric_prior'], source_key="ray_source", target_key="ray_target", ref_key="ct", sigma=cfg['sigma'], flip_lps_to_ras=True, prior_mode=cfg['prior_mode']),
    InjectEnergyDepositionFieldd(keys=['field'], spacing=cfg['pixdim']),
    EnsureTyped(keys=["ct","ray_source", "ray_target","geometric_prior","field"], track_meta=True )])

In [13]:
import json
data_list = json.load(open("data/unified.json"))

In [14]:
from monai.transforms import Invertd

inverse = Invertd(
    keys=["pred"],
    transform=infer_transforms,  # Pass the clean inference pipeline
    orig_keys=["ct"],            # Steal the spatial trace from the CT
    nearest_interp=False,        
    to_tensor=True
)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
wrapped_model = DoseCallWrapper(model,transforms=infer_transforms,inverse=inverse,device=device)

In [32]:
dict(data_list[0])

{'ct': '/scratch/db/proton/training/1ABB161/image/ct.mha',
 'gt_dose': '/scratch/db/proton/training/1ABB161/dose/Dose_B0_R0_L0.mha',
 'ray_source': [-4.39, -1020.17, -43.85],
 'ray_target': [-4.39, -20.16999999999996, -43.85],
 'condition': [0.0, 1.0, 80.5337]}

In [24]:
pred = wrapped_model(data_list[1500:1503])

Setting affine, but the applied meta contains an affine. This will be overwritten.
